> Instrukcja wykonania projektu
# Projekt Apache Spark

> Instrukcja wykonania projektu
# Wprowadzenie

Wykorzystując ten notatnik jako szablon zrealizuj projekt *Apache Spark* zgodnie z przydzielonym zestawem. 

Kilka uwag:

* W szablonie nie wolno zmieniać paragrafów *markdown*, zawierających *instrukcje wykonania projektu*. <br>
  Każdy taki paragraf jest oznaczony za pomocą `> Instrukcja wykonania projektu`
* Istniejące paragrafy zawierające *kod* uzupełnij w razie potrzeby zgodnie z instrukcjami
    - nie usuwaj ich
    - modyfikuj je tylko w zakresie zgodnym z instrukcjami
* Możesz dodawać nowe paragrafy zarówno zawierające kod jak i komentarze dotyczące tego kodu (markdown)
* Nie możesz zmieniać kolejności paragrafów zawierających instrukcje. Notatnik ma mieć niezmienioną strukturę.
* Utworzony notatnik musi być "samowystarczalny" w kontekście wykorzystywanej platformy przy założeniu dostepu do sieci.<br>
  Nie może wykorzystywać komponentów, które: 
    -  nie są domyślnie dostepne w ramach platformy i/lub
    -  nie są samodzielnie pobierane przez ten notatnik z oficjalnych repozytoriów (*Maven").  

> Instrukcja wykonania projektu

Poniżej w paragrafie markdown umieść numer oraz pełny tytuł przydzielonego zestawu

# Zestaw 23 – cars-rentals

> Instrukcja wykonania projektu

W ponizszym paragrafie określ wartość zmiennej `input_dir` wskazującej miejsce, w którym znajdują się Twoje dane źródłowe (katalogi `datasource1` oraz `datasource4`).

Ze zmiennej tej należy korzystać we wszystkich miejscach, w których odwołujemy się do danych źródłowych. Pełni ona rolę "parametru" naszego notatnika. 

***Pamiętaj*** aby przed rejestracją rozwiązania usunąć zawartość tej zmiennej. Osoba korzystająca z tego notatnika (np podczas oceny) samodzielnie ustawi sobie jej wartość na odpowiednią dla swojej konfiguracji.

In [ ]:
input_dir = ""

> Instrukcja wykonania projektu

# Konteksty
W poniższych paragrafach utwórz odpowiednie konteksty. Nasza aplikacja *Apache Spark* ma działać na klastrze z wykorzystaniem platformy *Hadoop (YARN)*. 

## Dodatkowe działania 

W zależności od potrzeb, wykorzystaj poniższe paragrafy do wykonania dodatkowych działań niezbędnych do utworzenia kontekstu.

> Instrukcja wykonania projektu

## Kontekst dla *DataFrame API*

Wykorzystaj poniższy paragraf do utworzenia obiektu kontekstu dla *DataFrame API*. 

Nazwij go `spark`

In [ ]:
from pyspark.sql import SparkSession
import os

spark = SparkSession.builder \
    .appName("RentalCars") \
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.3.2")\
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

> Instrukcja wykonania projektu

## Kontekst dla *RDD API*

Wykorzystaj poniższy paragraf do utworzenia obiektu kontekstu dla *RDD API*. 

Nazwij go `sc`. 

Utwórz go z powyżej utworzonego kontekstu. Dzięki temu całość notatnika będzie działała w ramach jednej aplikacji *Apache Spark*

In [ ]:
sc = spark.sparkContext

> Instrukcja wykonania projektu

# Część 1 - Spark Core (*RDD API*)

Pamiętaj o czystości Twojego API. Musi być ona zachowana od samego początku implementacji (utworzenia kontekstu), poprzez definicje źródeł i transformacje, aż do samego końca czyli wygenerowanie ostatecznego wyniku. 

Wykorzystaj odpowiedni kontekst utworzony w początkowej części notatnika.

> Instrukcja wykonania projektu

## Dane źródłowe

W poniższych paragrafach utwórz dwa obiekty RDD, których zawartością będą Twoje dane źródłowe. 

Użyj nazw czytelnie odnoszących się zawartości Twoich danych źródłowych. 

In [ ]:
datasource1_path = input_dir + "/datasource1"
datasource1 = sc.textFile(datasource1_path)

In [ ]:
datasource4_path = input_dir + "/datasource4"
datasource4 = sc.textFile(datasource4_path)

> Instrukcja wykonania projektu

## Przetwarzanie - etap 1 - "MapReduce" 

### Przetwarzanie

Twoim zadaniem jest utworzenie obiektu RDD o nazwie `mapreduce_RDD`, który w oparciu o: 

- utworzony kontekst, oraz
- pierwszy ze źródłowych RDD (`datasource1`)

będzie zawierał wynik zgodny z oczekiwaniami treści sekcji *Program MapReduce* dla Twojego zestawu. 

Wykorzystaj do tego celu poniższe paragrafy

In [ ]:
import csv
from io import StringIO
from pyspark.sql import Row

def map_function(line):
    if line.startswith("rental_id"):
        return None
    try:
        reader = csv.reader(StringIO(line))
        rental_id, car_id, customer_id, rental_start, rental_end, price, status = next(reader)
        year = rental_start.split("-")[0]
        completed_flag = 1 if status.strip().lower() == "completed" else 0
        return ((car_id, year), (1, completed_flag))
    except Exception:
        return None

def reduce_function(v1, v2):
    return (v1[0] + v2[0], v1[1] + v2[1])

def final_format(key_value):
    (car_id, year), (total_rentals, total_completed) = key_value
    
    if total_rentals > 0:
        completed_ratio = f"{total_completed / total_rentals:.2f}"
    else:
        completed_ratio = "0.00"
        
    return (f"{car_id},{year}", f"{total_rentals},{completed_ratio}")

# data1 to RDD wczytane z datasource1
map_RDD = datasource1.map(map_function).filter(lambda x: x is not None)

aggregated_RDD = map_RDD.reduceByKey(reduce_function)

mapreduce_RDD = aggregated_RDD.map(final_format)

> Instrukcja wykonania projektu

### Wynik

Pobierz liczbę elementów z `mapreduce_RDD`. 

Wykorzystaj do tego celu poniższy paragraf.

In [ ]:
count_mapreduce = mapreduce_RDD.count()
print(f"Liczba elementów w mapreduce_RDD: {count_mapreduce}")

> Instrukcja wykonania projektu

Posortuj rosnąco elementy `mapreduce_RDD` względem wszystkich wymaganych wynikowych atrybutów (w kolejności określonej w treści sekcji *Program MapReduce* dla Twojego zestawu). 

A następnie pobierz i wyświetl w oddzielnych liniach 10 pierwszych elementów. 

Wykorzystaj do tego celu poniższy paragraf.

In [ ]:
import operator

def parse_for_sort(line):
    try:
        key, value = line
        car_id, year = key.split(',')
        total_rentals_str, completed_ratio_str = value.split(',')
        
        car_id = car_id.strip()
        year = year.strip()
        total_rentals = int(total_rentals_str.strip())
        completed_ratio = float(completed_ratio_str.strip())
        
        sort_key = (car_id, year, total_rentals, completed_ratio)
        
        return (sort_key, line)
    except Exception:
        return None

sorted_mapreduce_RDD = mapreduce_RDD \
    .map(parse_for_sort) \
    .filter(lambda x: x is not None) \
    .sortBy(operator.itemgetter(0), ascending=True)

top_10_sorted = sorted_mapreduce_RDD.take(10)

print("--- 10 pierwszych, posortowanych elementów: ---")
for sort_key, original_line in top_10_sorted:
    print(f"{original_line[0]}\t{original_line[1]}")

> Instrukcja wykonania projektu

## Przetwarzanie - etap 2 - "Hive" 

### Przetwarzanie

Twoim zadaniem jest utworzenie obiektu RDD o nazwie `hive_RDD`, który w oparciu o: 

- utworzony powyżej `mapreduce_RDD` 
- drugi ze źródłowych RDD (`datasource4`)

będzie zawierał wynik zgodny z oczekiwaniami treści z sekcji *Program Hive* dla Twojego zestawu. 

Wykorzystaj do tego celu poniższe paragrafy

In [ ]:
import csv
import json
import operator
from io import StringIO

def parse_cars_for_rdd(line):
    if line.startswith("car_id"):
        return []  # Pomijanie nagłówka
    try:
        reader = csv.reader(StringIO(line))
        # car_id, make, model, year, features, category
        car_id, _, _, _, features, _ = next(reader)
        
        # Tworzenie krotek (car_id, feature) dla każdej cechy
        # Użycie strip() do usunięcia białych znaków, jeśli istnieją
        features_list = [f.strip() for f in features.split(';') if f.strip()]
        
        # Zwracamy klucz: car_id, wartość: feature
        return [(car_id.strip(), feature) for feature in features_list]
    except Exception:
        return []

# RDD par (car_id, feature)
cars_features_rdd = datasource4.flatMap(parse_cars_for_rdd)

# 1. Przygotowanie danych MapReduce (Key: car_id, Value: (year, rentals, ratio))
def prepare_mapreduce_data(kv):
    try:
        key, value = kv
        car_id, year_str = key.split(',')
        rentals_str, ratio_str = value.split(',')
        
        # Key: car_id | Value: (year, rentals_count, completion_ratio)
        return (car_id.strip(), 
                (int(year_str.strip()), int(rentals_str.strip()), float(ratio_str.strip())))
    except Exception:
        return None

# Zakładamy, że mapreduce_RDD jest dostępne
rentals_prepared_rdd = mapreduce_RDD.map(prepare_mapreduce_data).filter(lambda x: x is not None)


# 2. Łączenie (JOIN) RDD (po car_id)
# Wynik: (car_id, (feature, (year, rentals_count, completion_ratio)))
joined_rdd = cars_features_rdd.join(rentals_prepared_rdd)

# Mapowanie na klucz grupowania (feature, year) | Value: (rentals_count, completion_ratio, 1)
def prepare_for_feature_year_agg(kv):
    car_id, (feature, (year, rentals, ratio)) = kv
    # Klucz: (feature, year) | Value: (rentals, ratio, count)
    return ((feature, year), (rentals, ratio, 1))

feature_year_rdd = joined_rdd.map(prepare_for_feature_year_agg)


# 3. Agregacja (GROUP BY feature, year) - Obliczenie średnich
def aggregate_feature_year(v1, v2):
    # Suma: (total_rentals, total_ratio_sum, total_count)
    return (v1[0] + v2[0], v1[1] + v2[1], v1[2] + v2[2])

def calculate_feature_year_avg(kv):
    (feature, year), (total_rentals, total_ratio_sum, total_count) = kv
    
    avg_rentals = total_rentals / total_count
    avg_ratio = total_ratio_sum / total_count
    
    # Klucz: feature | Value: (year, avg_rentals, avg_ratio, avg_rentals_per_year)
    return (feature, (year, avg_rentals, avg_ratio, avg_rentals))

# Wynik: (feature, (year, avg_rentals, avg_ratio, avg_rentals_per_year))
feature_stats_rdd = feature_year_rdd.reduceByKey(aggregate_feature_year).map(calculate_feature_year_avg)


# 4. Agregacja globalna (GROUP BY feature) - Średnia dla wszystkich lat
def prepare_for_global_avg(kv):
    feature, (year, avg_rentals, avg_ratio, rentals_per_year) = kv
    # Klucz: feature | Value: (rentals_per_year, 1)
    return (feature, (rentals_per_year, 1))

def aggregate_global_avg(v1, v2):
    # Suma: (total_rentals_sum_of_years, total_years_count)
    return (v1[0] + v2[0], v1[1] + v2[1])

def calculate_global_avg(kv):
    feature, (total_rentals_sum_of_years, total_years_count) = kv
    global_avg = total_rentals_sum_of_years / total_years_count
    return (feature, global_avg)

# Wynik: (feature, global_avg_rentals)
feature_global_avg_rdd = feature_stats_rdd.map(prepare_for_global_avg).reduceByKey(aggregate_global_avg).map(calculate_global_avg)


# 5. Końcowe Łączenie i Formatowanie
final_joined_rdd = feature_stats_rdd.join(feature_global_avg_rdd)

def format_final_hive_rdd(kv):
    feature, ((year, avg_rentals, avg_ratio, rentals_per_year), global_avg) = kv
    
    # Warunek 'above_avg_rentals'
    above_avg = "true" if rentals_per_year > global_avg else "false"
    
    # Tworzenie obiektu (słownika) Pythona z zaokrąglonymi wartościami Float
    json_output = {
        "feature": feature,
        "rental_year": str(year),
        "feature_year_avg_rentals": f"{avg_rentals:.2f}",
        "feature_year_avg_completed_ratio": f"{avg_ratio:.2f}",
        "above_avg_rentals": above_avg
    }
    return json_output

# Końcowy RDD zawierający słowniki Pythona, gotowy do zapisu w formacie Pickle
hive_RDD = final_joined_rdd.map(format_final_hive_rdd)

> Instrukcja wykonania projektu

### Zapis wyniku 

Mając obiekt RDD o nazwie `hive_RDD` dokonaj jego zapisu w docelowym miejscu określonym w opisie projektu dla tej części projektu.

*Uwaga!* uwzględnij fakt, że docelowe miejsce może być "zajęte". 
Przed utworzeniem nowego wyniku usuń ewentualne efekty poprzednich uruchomień notatnika.  
Zwróć uwagę, że efekty te mogą istnieć na wielu poziomach.

#### Usunięcie poprzednich wyników

Wykonaj operacje usuwające ewentualne poprzednio utworzone wyniki.

Wykorzystaj do tego celu poniższe paragrafy

In [ ]:
import os
from subprocess import run, CalledProcessError
import json

# Definicja docelowego katalogu wyjściowego
output_rdd_part1 = "/tmp/output1"

# KROK USUWANIA
print(f"Sprawdzam i usuwam istniejący katalog HDFS: {output_rdd_part1}")

try:
    run(["hdfs", "dfs", "-rm", "-r", output_rdd_part1], capture_output=True, text=True, check=False)
    print("Próba usunięcia katalogu HDFS zakończona (może zgłaszać błąd, jeśli nie istniał).")
except FileNotFoundError:
    print("Ostrzeżenie: Nie znaleziono komendy 'hdfs'. Ręczne usuwanie może być konieczne.")
except Exception as e:
    print(f"Wystąpił nieoczekiwany błąd podczas usuwania katalogu HDFS: {e}")

> Instrukcja wykonania projektu

#### Zapis nowe wyniki

Dokonaj zapisu nowego wyniku.

Wykorzystaj do tego celu poniższe paragrafy

In [ ]:
# KROK ZAPISU
print(f"\nRozpoczynam zapis hive_RDD w formacie Pickle do: {output_rdd_part1}")

def format_final_hive_rdd_for_pickle(kv):
    feature, ((year, avg_rentals, avg_ratio, rentals_per_year), global_avg) = kv
    
    above_avg = "true" if rentals_per_year > global_avg else "false"
    
    # Zwrócenie krotki
    return (
        feature,
        year,
        float(f"{avg_rentals:.2f}"),
        float(f"{avg_ratio:.2f}"),
        above_avg
    )

# Nowy RDD zawierający krotki, gotowy do zapisu w formacie Pickle
pickle_ready_RDD = final_joined_rdd.map(format_final_hive_rdd_for_pickle)

# --- ZAPIS ---
pickle_ready_RDD.saveAsPickleFile(output_rdd_part1)

> Instrukcja wykonania projektu

### Wynik 

Odczytaj do zmiennej `result_RDD` dane z docelowego miejsca, w którym została zapisana zawartość `hive_RDD`.

Wykorzystaj do tego celu poniższe paragrafy

In [ ]:
# Odczyt danych
try:
    result_RDD = sc.pickleFile(output_rdd_part1)
except Exception as e:
    print(f"BŁĄD ODCZYTU RDD: Nie udało się wczytać pliku Pickle z {output_rdd_part1}. {e}")

> Instrukcja wykonania projektu

Pobierz liczbę elementów z `result_RDD`. 

Wykorzystaj do tego celu poniższy paragraf.

In [ ]:
count_result_RDD = result_RDD.count()
print(f"Liczba elementów w result_RDD: {count_result_RDD}")

> Instrukcja wykonania projektu

Posortuj rosnąco elementy `result_RDD` względem wszystkich wymaganych wynikowych atrybutów (w kolejności określonej w treści sekcji *Program Hive* dla Twojego zestawu). 

A następnie pobierz i wyświetl w oddzielnych liniach 10 pierwszych elementów. 

Wykorzystaj do tego celu poniższy paragraf.

In [ ]:
import operator

# Funkcja do przygotowania klucza sortowania z wczytanej kortki
def prepare_sort_key_from_tuple(data):
    if not isinstance(data, tuple) or len(data) != 5:
        return None # Odrzucenie niepoprawnych krotek
    
    try:
        # Ekstrakcja danych według indeksu
        feature = data[0]
        rental_year = data[1]
        avg_rentals = data[2]
        avg_completed_ratio = data[3]
        above_avg = data[4]
        
        # Klucz sortowania
        sort_key = (
            feature,
            rental_year,
            avg_rentals,
            avg_completed_ratio,
            above_avg 
        )
        
        # Zwracamy klucz sortowania i oryginalną krotkę
        return (sort_key, data)
        
    except Exception:
        return None

# 1. Parsowanie i sortowanie
sorted_result_RDD = result_RDD \
    .map(prepare_sort_key_from_tuple) \
    .filter(lambda x: x is not None) \
    .sortBy(operator.itemgetter(0), ascending=True)

# 2. Pobranie i wyświetlenie 10 pierwszych elementów
top_10_sorted = sorted_result_RDD.take(10)

print("10 pierwszych, posortowanych elementów")
# Wyświetlamy tylko oryginalny obiekt (element 1 krotki)
for sort_key, original_tuple in top_10_sorted:
    print(original_tuple)

> Instrukcja wykonania projektu

# Część 2 - Spark SQL (*DataFrame API*)

Pamiętaj o czystości Twojego API. Musi być ona zachowana od samego początku implementacji (utworzenia kontekstu), poprzez definicje źródeł i transformacje, aż do samego końca czyli wygenerowanie ostatecznego wyniku. 

Wykorzystaj odpowiedni kontekst utworzony w początkowej części notatnika.

> Instrukcja wykonania projektu

## Dane źródłowe

W poniższych paragrafach utwórz dwa obiekty `DataFrame`, których zawartością będą Twoje dane źródłowe. 

Użyj nazw czytelnie odnoszących się zawartości Twoich danych źródłowych. 

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
from pyspark.sql.functions import col

# Definicja Schematu dla datasource1
rentals_schema = StructType([
    StructField("rental_id", StringType(), True),
    StructField("car_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("rental_start", StringType(), True),
    StructField("rental_end", StringType(), True),
    StructField("price", StringType(), True),
    StructField("status", StringType(), True)
])

# Wczytanie data1
rentals_df = spark.read.csv(
    datasource1,
    sep=',',
    header=True, # Zakładamy, że pierwszy wiersz to nagłówek
    schema=rentals_schema
)

# Definicja Schematu dla datasource4
cars_schema = StructType([
    StructField("car_id", StringType(), True),
    StructField("make", StringType(), True),
    StructField("model", StringType(), True),
    StructField("year", StringType(), True),
    StructField("features", StringType(), True),
    StructField("category", StringType(), True)
])

# Wczytanie data4
cars_df = spark.read.csv(
    datasource4,
    sep=',',
    header=True, # Zakładamy, że pierwszy wiersz to nagłówek
    schema=cars_schema
)

> Instrukcja wykonania projektu

## Przetwarzanie - etap 1 - "MapReduce" 

### Przetwarzanie

Twoim zadaniem jest utworzenie obiektu `DataFrame` o nazwie `mapreduce_DF`, który w oparciu o: 

- utworzony kontekst, oraz
- pierwszy ze źródłowych RDD (`datasource1`)

będzie zawierał wynik zgodny z oczekiwaniami treści sekcji *Program MapReduce* dla Twojego zestawu. 

Wykorzystaj do tego celu poniższe paragrafy

In [ ]:
from pyspark.sql.functions import col, year, when, count, sum, round, lit, regexp_extract

# 1. Transformacja i przygotowanie kolumn (Mapper)
transformed_df = rentals_df.select(
    col("car_id"),
    # Ekstrakcja roku z 'rental_start'
    regexp_extract(col("rental_start"), r"^(\d{4})-\d{2}-\d{2}", 1).alias("rental_year"),
    # Flaga ukończenia
    when(col("status").ilike("completed"), lit(1)).otherwise(lit(0)).alias("completed_flag")
) \
.filter(col("car_id").isNotNull()) # Pomijanie wierszy z brakującym kluczem

# 2. Agregacja (Combiner/Reducer)
mapreduce_DF_aggregated = transformed_df.groupBy("car_id", "rental_year") \
    .agg(
        count(lit(1)).alias("total_rentals"),
        sum("completed_flag").alias("total_completed")
    )

# 3. Obliczenie współczynnika ukończenia
mapreduce_DF = mapreduce_DF_aggregated.withColumn(
    "completed_ratio",
    round(col("total_completed") / col("total_rentals"), 2)
) \
.select(
    "car_id",
    col("rental_year").cast(StringType()),
    "total_rentals",
    "completed_ratio"
)

> Instrukcja wykonania projektu

### Wynik

Pobierz liczbę elementów z `mapreduce_DF`. 

Wykorzystaj do tego celu poniższy paragraf.

In [ ]:
count_mapreduce_DF = mapreduce_DF.count()
print(f"Liczba elementów w mapreduce_DF: {count_mapreduce_DF}")

> Instrukcja wykonania projektu

Posortuj rosnąco elementy `mapreduce_DF` względem wszystkich wymaganych wynikowych atrybutów (w kolejności określonej w treści sekcji *Program MapReduce* dla Twojego zestawu). 

A następnie pobierz i wyświetl w oddzielnych liniach 10 pierwszych elementów. Skorzystaj z metody `show()`.

Wykorzystaj do tego celu poniższy paragraf.

In [ ]:
from pyspark.sql.functions import col

# Sortowanie DataFrame rosnąco po wszystkich kolumnach wynikowych.
# Domyślnie sortowanie w orderBy() jest rosnące (asc).
mapreduce_DF_sorted = mapreduce_DF.orderBy(
    col("car_id").asc(),
    col("rental_year").asc(),
    col("total_rentals").asc(),
    col("completed_ratio").asc()
)

print("10 pierwszych, posortowanych elementów mapreduce_DF:")
mapreduce_DF_sorted.show(10)

> Instrukcja wykonania projektu

## Przetwarzanie - etap 2 - "Hive" 

### Przetwarzanie

Twoim zadaniem jest utworzenie obiektu `DataFrame`o nazwie `hive_DF`, który w oparciu o: 

- utworzony powyżej `mapreduce_DF` 
- drugi ze źródłowych `DataFrame` (`datasource4`)

będzie zawierał wynik zgodny z oczekiwaniami treści z sekcji *Program Hive* dla Twojego zestawu. 

Wykorzystaj do tego celu poniższe paragrafy

In [ ]:
from pyspark.sql.functions import col, explode, split, avg, round, lit, when, format_string, concat, trim

# 1. Rozbicie cech (cars_exploded) 
cars_exploded_df = cars_df \
    .filter((col("features").isNotNull()) & (col("features") != "")) \
    .withColumn("feature", explode(split(col("features"), ";"))) \
    .select(col("car_id"), trim(col("feature")).alias("feature"))

# 2. Łączenie danych
joined_data_df = mapreduce_DF.alias("r") \
    .join(cars_exploded_df.alias("e"), col("r.car_id") == col("e.car_id"), "inner") \
    .select(
        col("e.feature"),
        col("r.rental_year").cast("int").alias("rental_year"),
        col("r.total_rentals"),
        col("r.completed_ratio")
    )

# 3. Agregacja cech w danym roku (feature_stats)
feature_stats_df = joined_data_df \
    .groupBy("feature", "rental_year") \
    .agg(
        avg("total_rentals").alias("feature_year_avg_rentals"),
        avg("completed_ratio").alias("feature_year_avg_completed_ratio")
    )

# 4. Agregacja globalna dla każdej cechy
feature_avg_df = feature_stats_df \
    .groupBy("feature") \
    .agg(
        avg("feature_year_avg_rentals").alias("avg_rentals_all_years")
    )

# 5. Końcowe Łączenie i Obliczanie Flagi
final_result_df = feature_stats_df.alias("fs") \
    .join(feature_avg_df.alias("fa"), col("fs.feature") == col("fa.feature"), "inner") \
    .withColumn("above_avg_rentals", 
        # Obliczenie flagi
        when(col("fs.feature_year_avg_rentals") > col("fa.avg_rentals_all_years"), lit("true"))
        .otherwise(lit("false"))
    ) \
    .select(
        col("fs.feature"),
        col("fs.rental_year"),
        round(col("fs.feature_year_avg_rentals"), 2).alias("feature_year_avg_rentals"),
        round(col("fs.feature_year_avg_completed_ratio"), 2).alias("feature_year_avg_completed_ratio"),
        col("above_avg_rentals")
    )

> Instrukcja wykonania projektu

### Zapis wyniku 

Mając obiekt `DataFrame` o nazwie `hive_DF` dokonaj jego zapisu w docelowym miejscu określonym w opisie projektu dla tej części projektu.

*Uwaga!* uwzględnij fakt, że docelowe miejsce może być "zajęte". 
Przed utworzeniem nowego wyniku usuń ewentualne efekty poprzednich uruchomień notatnika.  
Zwróć uwagę, że efekty te mogą istnieć na wielu poziomach.

#### Usunięcie poprzednich wyników

Wykonaj operacje usuwające ewentualne poprzednio utworzone wyniki.

Wykorzystaj do tego celu poniższe paragrafy

In [ ]:
from subprocess import run, CalledProcessError

# 1. Definicja docelowego katalogu wyjściowego
output_df_part2 = "/tmp/output2" 

# KROK USUWANIA
print(f"Sprawdzam i usuwam istniejący katalog HDFS: {output_df_part2}")

try:
    run(["hdfs", "dfs", "-rm", "-r", output_df_part2], capture_output=True, text=True, check=False)
    print("Próba usunięcia katalogu HDFS zakończona (może zgłaszać błąd, jeśli nie istniał).")
except:
    pass

> Instrukcja wykonania projektu

#### Zapis nowe wyniki

Dokonaj zapisu nowego wyniku.

Wykorzystaj do tego celu poniższe paragrafy

In [ ]:
# KROK ZAPISU
print(f"\nRozpoczynam zapis strukturalnego final_result_df w formacie Delta Lake do: {output_df_part2}")

try:
    final_result_df.write \
        .mode("overwrite") \
        .format("delta") \
        .save(output_df_part2)
    print("Zapis DataFrame strukturalnego zakończony pomyślnie w formacie Delta Lake.")
except Exception as e:
    print(f"BŁĄD ZAPISU DATAFRAME: Wystąpił błąd podczas zapisu w formacie Delta. {e}")

> Instrukcja wykonania projektu

### Wynik 

Odczytaj do zmiennej `result_DF` dane z docelowego miejsca, w którym została zapisana zawartość `hive_DF`.

Wykorzystaj do tego celu poniższe paragrafy

In [ ]:
from pyspark.sql.functions import col

# Odczyt danych z katalogu
result_DF = spark.read \
    .format("delta") \
    .load(output_df_part2)

> Instrukcja wykonania projektu

Pobierz liczbę elementów z `result_DF`. 

Wykorzystaj do tego celu poniższy paragraf.

In [ ]:
count_result_DF = result_DF.count()
print(f"Liczba elementów w result_DF: {count_result_DF}")

> Instrukcja wykonania projektu

Posortuj rosnąco elementy `result_DF` względem wszystkich wymaganych wynikowych atrybutów (w kolejności określonej w treści sekcji *Program Hive* dla Twojego zestawu). 

A następnie pobierz i wyświetl w oddzielnych liniach 10 pierwszych elementów. 

Wykorzystaj do tego celu poniższy paragraf.

In [ ]:
from pyspark.sql.functions import col
print("--- 10 pierwszych, posortowanych elementów result_DF:")

# Sortowanie DataFrame rosnąco po kolumnach wynikowych
result_DF_sorted = result_DF.orderBy(
    col("feature").asc(),
    col("rental_year").asc(),
    col("feature_year_avg_rentals").asc(),
    col("feature_year_avg_completed_ratio").asc(),
    col("above_avg_rentals").asc()
)

result_DF_sorted.show(10)

> Instrukcja wykonania projektu

## Zamknięcie kontekstu 

Zamknij kontekst, wyłączając w ten sposób aplikację *Apache Spark*.

Wykorzystaj do tego celu poniższy paragraf.

In [ ]:
# Zatrzymanie SparkSession
spark.stop()

print("Apache Spark Session została pomyślnie zatrzymana i kontekst zamknięty.")

> Instrukcja wykonania projektu

# Podsumowanie 

Jeśli implementacja tego notatnika została przez Ciebie zakończona, koniecznie uruchom jego całość kilkukrotnie, aby upewnić się, że całość przetwarzania jest poprawna, a samo przetwarzanie jest powtarzalne. 

Na zakończenie, przed rejestracją tego notatnika w ramach projektu, pamiętaj o:

- usunięciu wartości zmiennej `input_dir`
- wyczyszczeniu wszystkich wyników (prawy klawisz myszy i pozycja *Clear Output of All Cells*). <br>
  Pozostawienie Twoich wyników może być potraktowane jako chęć wpływania na ocenę recenzentów
